# Trajectory Module

The `Trajectory` class provides lazy, memory-efficient reading of multi-frame LAMMPS dump files. It only reads frame data when you access a specific timestep.

Key features:
- **Lazy reading**: only the requested frame is loaded into memory
- **Slicing**: `traj[start:stop]` creates sub-trajectories without copying data
- **Concatenation**: `t1 + t2` joins timeslices
- **ASE integration**: `timeslice.to_atoms()` converts to list of ASE `Atoms`

In [ ]:
from pyscal import Trajectory

## Loading a Trajectory

Create a `Trajectory` object by passing the path to a LAMMPS dump file.

In [ ]:
traj = Trajectory("traj.light")
print(f"Trajectory: {traj.nblocks} frames, {traj.natoms} atoms per frame")
print(repr(traj))

## Accessing Frames

Access individual frames by index. Each index returns a `Timeslice` object. Call `.to_atoms()` to get a list of ASE `Atoms` objects.

In [ ]:
# Access first frame
ts = traj[0]
print(repr(ts))

# Convert to ASE Atoms (returns a list, one per frame in the slice)
atoms_list = ts.to_atoms()
atoms = atoms_list[0]
print(f"ASE Atoms: {len(atoms)} atoms, cell = {atoms.cell.lengths()}")

## Computing Descriptors on Trajectory Frames

In [ ]:
import pyscal

# Compute CNA for each frame
for i in range(traj.nblocks):
    ts = traj[i]
    atoms_list = ts.to_atoms()
    atoms = atoms_list[0]
    cna = pyscal.common_neighbor_analysis(atoms)
    print(f"Frame {i}: CNA = {cna}")

## Slicing Trajectories

In [ ]:
# Take a subset of frames
sub = traj[:2]
atoms_list = sub.to_atoms()
print(f"Sub-trajectory: {len(atoms_list)} frames")

## Writing Frames

You can write individual frames back to LAMMPS dump format.

In [ ]:
import tempfile, os

# Write first frame to a temporary file
ts = traj[0]
tmpfile = os.path.join(tempfile.gettempdir(), "test_frame.dump")
ts.to_file(tmpfile)

# Verify the file was written
size = os.path.getsize(tmpfile)
print(f"Wrote frame to {tmpfile} ({size} bytes)")
os.remove(tmpfile)

## Concatenating Timeslices

In [ ]:
# Join two sub-trajectories
t1 = traj[:1]
t2 = traj[1:]
combined = t1 + t2
atoms_list = combined.to_atoms()
print(f"Combined: {len(atoms_list)} frames")